<a href="https://colab.research.google.com/github/qufp/CSN-2026-/blob/main/Pandas_para_ciencia_de_datos_proyecto_I_parte_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import io
import pandas as pd
from google.colab import files
from IPython.display import display


# --------------------------------------------------
# 1. CARGAR LOS DATOS
# --------------------------------------------------

print("Selecciona el archivo retail_sales.csv.")
archivos = files.upload()

if len(archivos) != 1:
    raise ValueError("Selecciona un solo archivo CSV.")

nombre_archivo = list(archivos.keys())[0]

if not nombre_archivo.lower().endswith(".csv"):
    raise ValueError("El archivo debe tener extensión .csv.")

datos = pd.read_csv(io.BytesIO(archivos[nombre_archivo]))

# Eliminar posibles espacios al inicio o al final de los encabezados.
datos.columns = datos.columns.str.strip()

columnas_necesarias = ["Producto", "Tienda", "Ventas", "Precio"]
columnas_faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in datos.columns
]

if columnas_faltantes:
    raise ValueError(
        f"Faltan estas columnas: {', '.join(columnas_faltantes)}. "
        f"El archivo contiene: {', '.join(datos.columns)}. "
        "Comprueba que sea el dataset de esta actividad."
    )

print("\nPRIMERAS 10 FILAS")
display(datos.head(10))


# --------------------------------------------------
# 2. EXPLORACIÓN INICIAL
# --------------------------------------------------

print("\nÚLTIMAS 5 FILAS")
display(datos.tail(5))

# info() muestra la cantidad de registros, las columnas,
# sus tipos de datos y la memoria utilizada.
print("\nINFORMACIÓN GENERAL")
datos.info()

print("\nTIPOS DE DATOS ORIGINALES")
print(datos.dtypes)

# Revisar valores faltantes antes de realizar cambios.
print("\nVALORES FALTANTES POR COLUMNA")
print(datos.isna().sum())


# --------------------------------------------------
# 3. PREPARAR LAS COLUMNAS PARA EL ANÁLISIS
# --------------------------------------------------

# Quitar espacios de los textos y conservar los valores faltantes.
for columna in ["Producto", "Tienda"]:
    datos[columna] = datos[columna].astype("string").str.strip()
    datos[columna] = datos[columna].replace("", pd.NA)

# Convertir ventas y precios a números.
# Si algún valor no se puede convertir, quedará como NaN.
# No se reemplaza por cero porque eso cambiaría su significado.
for columna in ["Ventas", "Precio"]:
    valores_originales = datos[columna].copy()

    datos[columna] = pd.to_numeric(
        datos[columna],
        errors="coerce"
    )

    conversiones_invalidas = (
        valores_originales.notna() & datos[columna].isna()
    ).sum()

    print(
        f"\nValores no numéricos convertidos a NaN en {columna}: "
        f"{conversiones_invalidas}"
    )

print("\nTIPOS DE DATOS DESPUÉS DE LA PREPARACIÓN")
print(datos.dtypes)

# describe() resume las columnas numéricas.
# Los valores faltantes no participan en estos cálculos.
print("\nESTADÍSTICAS DESCRIPTIVAS")
display(datos.describe())

print("\nVALORES FALTANTES DESPUÉS DE LA PREPARACIÓN")
print(datos.isna().sum())


# --------------------------------------------------
# 4. INSPECCIÓN DE PRODUCTOS Y TIENDAS
# --------------------------------------------------

# Contar cuántas filas corresponden a cada producto.
# Este conteo no representa la suma de las ventas.
print("\nCANTIDAD DE REGISTROS POR PRODUCTO")
print(datos["Producto"].value_counts(dropna=False))

# Mostrar las tiendas sin repetirlas.
print("\nTIENDAS DISPONIBLES")
print(datos["Tienda"].unique())


# --------------------------------------------------
# 5. FILTRADO DE DATOS
# --------------------------------------------------

# Seleccionar filas donde Ventas sea mayor a 50.
ventas_mayores_50 = datos[datos["Ventas"] > 50]

print("\nVENTAS MAYORES A 50")
print("Cantidad de registros:", len(ventas_mayores_50))
display(ventas_mayores_50)

# Seleccionar filas donde Precio sea menor a 0.5.
precios_menores_05 = datos[datos["Precio"] < 0.5]

print("\nPRECIOS MENORES A 0.5")
print("Cantidad de registros:", len(precios_menores_05))
display(precios_menores_05)

# query() permite escribir las condiciones dentro de un texto.
# El nombre Manzanas debe coincidir con el texto del archivo.
ventas_manzanas = datos.query(
    "Producto == 'Manzanas' and Ventas > 30",
    engine="python"
)

print("\nMANZANAS CON VENTAS MAYORES A 30")
print("Cantidad de registros:", len(ventas_manzanas))
display(ventas_manzanas)


# --------------------------------------------------
# 6. SELECCIÓN DE FILAS Y COLUMNAS
# --------------------------------------------------

# Seleccionar únicamente las columnas Producto y Ventas.
print("\nCOLUMNAS PRODUCTO Y VENTAS")
display(datos[["Producto", "Ventas"]])

# loc selecciona por etiquetas del índice e incluye ambos límites.
# Con el índice original, selecciona las etiquetas 5, 6, 7, 8, 9 y 10.
# Como Python comienza en 0, son los registros sexto al undécimo.
print("\nÍNDICES DEL 5 AL 10: PRODUCTO Y TIENDA")
display(datos.loc[5:10, ["Producto", "Tienda"]])

# iloc selecciona por posición y no incluye el límite final.
# :5 selecciona las primeras cinco filas.
# :3 selecciona las primeras tres columnas.
print("\nPRIMERAS 5 FILAS Y PRIMERAS 3 COLUMNAS")
display(datos.iloc[:5, :3])

print("\nAnálisis exploratorio finalizado.")
